# Notebook 04: Preprocessing & Discretization Pipeline
**Project:** Mining Sequential Activation Patterns for Early Hallucination Detection in LLMs  
**Role:** Member 3 (Preprocessing & Discretization)  
**Deliverables:** Preprocessed activation matrices, fitted transformation models (Scaler, PCA, KMeans), and temporally ordered discrete sequence datasets for PrefixSpan pattern mining.

---

### Pipeline Architecture:
$$\text{Raw Activations } (T \times D) \xrightarrow{\text{Cleaning}} \mathbf{X} \xrightarrow{\text{StandardScaler}} \mathbf{X}_{\text{norm}} \xrightarrow{\text{PCA (Optional)}} \mathbf{X}_{\text{proj}} \xrightarrow{\text{K-Means } (K=20)} \text{State IDs } s_t \xrightarrow{\text{Group by Prompt}} \langle s_0, s_1, \dots, s_{T-1} \rangle$$


In [1]:
import os
import sys
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Add project root to sys.path
sys.path.append(os.path.abspath('..'))

from src.preprocessing import (
    load_raw_activations,
    clean_and_prepare_dataset,
    ActivationPreprocessor
)
from src.clustering import KMeansDiscretizer
from src.sequence_builder import (
    build_ordered_sequences,
    export_sequences
)

print("All modules imported successfully!")


All modules imported successfully!


## 1. Data Loading & Cleaning
In this section, we load raw internal activation tensors (`.pt` files) generated during token generation by the LLM (Qwen2.5-1.5B-Instruct).
Each file contains activation matrices across layers `[10, 18, 26]` of hidden dimension 1536.
We validate:
- Record completeness and absence of `NaN` or `Inf`
- Alignment with response-level labels (`0 = Truthful`, `1 = Hallucinated`) from `labeled_generated_responses_50.csv`


In [2]:
ACTIVATIONS_DIR = "../data/activations"
ZIP_FALLBACK = "../activations_50.zip"
LABELS_CSV = "../data/processed/labeled_generated_responses_50.csv"
SELECTED_LAYER = 18  # Default middle/late layer

raw_records = load_raw_activations(
    activations_dir=ACTIVATIONS_DIR,
    zip_fallback=ZIP_FALLBACK
)

X_raw, tokens_df, prompts_df = clean_and_prepare_dataset(
    raw_records=raw_records,
    labels_csv_path=LABELS_CSV,
    selected_layer=SELECTED_LAYER
)

print(f"\nExtracted raw feature matrix shape: {X_raw.shape}")
prompts_df.head(3)


Loaded 50 raw activation files from '../data/activations'.
[Data Cleaning Summary]
  Valid Prompts: 50 / 50
  Truthful Prompts (Label 0): 25
  Hallucinated Prompts (Label 1): 25
  Total Token Vectors Extracted: 2553
  Vector Dimensionality: 1536

Extracted raw feature matrix shape: (2553, 1536)


## 2. Normalization & Dimensionality Reduction (PCA)
Neural activations often exhibit heterogeneous variance across feature dimensions. We apply `StandardScaler` to zero-center and standardize features.
We also provide PCA dimensionality reduction (keeping it toggleable).


In [3]:
USE_PCA = False  # Set to True to enable PCA dimensionality reduction
PCA_COMPONENTS = 0.95  # Retain 95% variance if PCA is enabled

preprocessor = ActivationPreprocessor(
    normalize=True,
    use_pca=USE_PCA,
    pca_components=PCA_COMPONENTS,
    random_state=42
)

X_processed = preprocessor.fit_transform(X_raw)

# Save fitted preprocessors
os.makedirs("../models", exist_ok=True)
saved_paths = preprocessor.save(output_dir="../models")
print(f"Saved preprocessor artifacts: {saved_paths}")
print(f"Processed matrix shape ready for clustering: {X_processed.shape}")


Saved preprocessor artifacts: {'scaler': '../models\\scaler.joblib'}
Processed matrix shape ready for clustering: (2553, 1536)


### Diagnostic: PCA Variance Analysis
Let us inspect the cumulative explained variance across principal components to evaluate dimensionality structure.


In [4]:
from sklearn.decomposition import PCA

pca_diag = PCA(n_components=min(100, X_raw.shape[0], X_raw.shape[1]), random_state=42)
pca_diag.fit(X_processed if not USE_PCA else preprocessor.scaler.transform(X_raw))
cum_var = np.cumsum(pca_diag.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker='o', markersize=3, color='#1f77b4', linewidth=1.5)
plt.axhline(y=0.90, color='r', linestyle='--', label='90% Explained Variance')
plt.axhline(y=0.95, color='g', linestyle='--', label='95% Explained Variance')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Activation Vector Dimensionality: PCA Cumulative Variance')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


## 3. K-Means Discretization ($K = 20$)
We cluster continuous token activation vectors into $K = 20$ discrete states.
Each cluster ID represents a distinct recurring internal state of the model during token generation.


In [5]:
K_CLUSTERS = 20

discretizer = KMeansDiscretizer(
    n_clusters=K_CLUSTERS,
    random_state=42,
    n_init=10
)

cluster_labels = discretizer.fit_predict(X_processed)
metrics = discretizer.evaluate(X_processed, cluster_labels)

print("=" * 50)
print(f"K-Means (K={K_CLUSTERS}) Clustering Quality Metrics:")
print("=" * 50)
print(f"  Inertia:                 {metrics['inertia']:,.2f}")
print(f"  Silhouette Score:        {metrics['silhouette_score']:.4f}")
print(f"  Davies-Bouldin Index:    {metrics['davies_bouldin_index']:.4f} (lower is better)")
print(f"  Calinski-Harabasz Index: {metrics['calinski_harabasz_index']:,.2f} (higher is better)")
print(f"  Min Cluster Size:        {metrics['min_cluster_size']}")
print(f"  Max Cluster Size:        {metrics['max_cluster_size']}")
print(f"  Mean Cluster Size:       {metrics['mean_cluster_size']:.1f}")

# Save fitted KMeans model
model_path = discretizer.save("../models/kmeans_k20.joblib")


K-Means (K=20) Clustering Quality Metrics:
  Inertia:                 3,395,800.75
  Silhouette Score:        0.0181
  Davies-Bouldin Index:    4.4169 (lower is better)
  Calinski-Harabasz Index: 20.63 (higher is better)
  Min Cluster Size:        15
  Max Cluster Size:        426
  Mean Cluster Size:       127.7
Saved KMeans model to '../models/kmeans_k20.joblib'.


In [6]:
# Visualize Cluster Distribution
cluster_ids = list(metrics['cluster_distribution'].keys())
counts = list(metrics['cluster_distribution'].values())

plt.figure(figsize=(10, 4))
bars = plt.bar(cluster_ids, counts, color='#2ca02c', edgecolor='black', alpha=0.8)
plt.xlabel('Cluster / State ID')
plt.ylabel('Number of Assigned Token Vectors')
plt.title(f'Cluster Size Distribution (K={K_CLUSTERS}, Total Tokens={len(cluster_labels)})')
plt.xticks(cluster_ids)
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


## 4. Temporal Sequence Construction
We map each token to its cluster ID, group by `Prompt_ID`, and sort chronologically by generation step (`Token_Step`).
This produces discrete sequences representing internal trajectories:
$$\langle s_0, s_1, s_2, \dots, s_{T-1} \rangle$$


In [7]:
sequences_df = build_ordered_sequences(
    tokens_df=tokens_df,
    prompts_df=prompts_df,
    cluster_labels=cluster_labels
)

print("\nFirst 5 Constructed Sequences:")
sequences_df[['Prompt_ID', 'Label_Name', 'Sequence_Length', 'Sequence_Str']].head(5)


[Sequence Construction Summary]
  Total Sequences Built: 50
  Truthful Sequences (0): 25
  Hallucinated Sequences (1): 25
  Min Length: 4
  Max Length: 80
  Mean Length: 51.06

First 5 Constructed Sequences:


## 5. Exploratory Analysis of Sequences
We examine whether truthful and hallucinated responses differ in sequence lengths or initial cluster distribution.


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Subplot 1: Sequence Length Distribution
truthful_lens = sequences_df[sequences_df['Response_Label'] == 0]['Sequence_Length']
hallucinated_lens = sequences_df[sequences_df['Response_Label'] == 1]['Sequence_Length']

axes[0].hist(truthful_lens, bins=10, alpha=0.6, label='Truthful (0)', color='blue', edgecolor='black')
axes[0].hist(hallucinated_lens, bins=10, alpha=0.6, label='Hallucinated (1)', color='red', edgecolor='black')
axes[0].set_xlabel('Sequence Length (Tokens)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Sequence Length by Label')
axes[0].legend()
axes[0].grid(True, linestyle=':', alpha=0.6)

# Subplot 2: Initial State Distribution (Token 0)
initial_states_truthful = [seq[0] for seq in sequences_df[sequences_df['Response_Label'] == 0]['Sequence']]
initial_states_hallucinated = [seq[0] for seq in sequences_df[sequences_df['Response_Label'] == 1]['Sequence']]

axes[1].hist(initial_states_truthful, bins=range(K_CLUSTERS + 1), alpha=0.6, label='Truthful (0)', color='blue', align='left')
axes[1].hist(initial_states_hallucinated, bins=range(K_CLUSTERS + 1), alpha=0.6, label='Hallucinated (1)', color='red', align='left')
axes[1].set_xlabel('Initial State ID (Token 0)')
axes[1].set_ylabel('Count')
axes[1].set_title('Initial Activation State Distribution')
axes[1].set_xticks(range(K_CLUSTERS))
axes[1].legend()
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


## 6. Exporting Datasets for PrefixSpan Sequential Pattern Mining
We export the final discrete sequences into `data/processed/sequences/` in multiple standard formats:
1. **Tabular CSV**: `sequences_k20.csv` containing full metadata, prompt text, response text, and sequences.
2. **Structured JSON**: `sequences_k20.json` for easy loading in downstream Python notebooks.
3. **SPMF Format**: Standard space-separated sequence format for SPMF/PrefixSpan pattern mining tools (`truthful_sequences_spmf.txt` and `hallucinated_sequences_spmf.txt`).


In [9]:
OUTPUT_SEQUENCES_DIR = "../data/processed/sequences"

exported_files = export_sequences(
    sequences_df=sequences_df,
    output_dir=OUTPUT_SEQUENCES_DIR,
    prefix="sequences_k20"
)

print("\nVerification of Exported Sequence Files:")
for key, fpath in exported_files.items():
    print(f"  [{key.upper()}] {fpath} (Exists: {os.path.exists(fpath)}, Size: {os.path.getsize(fpath):,} bytes)")


Exported tabular sequences to: '../data/processed/sequences\sequences_k20.csv'
Exported JSON sequences to: '../data/processed/sequences\sequences_k20.json'
Exported SPMF format files:
  All: '../data/processed/sequences\sequences_k20_all_spmf.txt'
  Truthful (0): '../data/processed/sequences\truthful_sequences_spmf.txt' (25 sequences)
  Hallucinated (1): '../data/processed/sequences\hallucinated_sequences_spmf.txt' (25 sequences)

Verification of Exported Sequence Files:
  [CSV] ../data/processed/sequences\sequences_k20.csv (Exists: True, Size: 46,484 bytes)
  [JSON] ../data/processed/sequences\sequences_k20.json (Exists: True, Size: 62,977 bytes)
  [SPMF_ALL] ../data/processed/sequences\sequences_k20_all_spmf.txt (Exists: True, Size: 14,184 bytes)
  [SPMF_TRUTHFUL] ../data/processed/sequences\truthful_sequences_spmf.txt (Exists: True, Size: 7,568 bytes)
  [SPMF_HALLUCINATED] ../data/processed/sequences\hallucinated_sequences_spmf.txt (Exists: True, Size: 6,616 bytes)


## Summary & Handoff to Member 4
- **Member 3 deliverables complete:**
  - Standardized activation feature matrix (2,553 token vectors across 50 prompts).
  - Trained K-Means ($K=20$) model saved in `models/kmeans_k20.joblib`.
  - Serialized preprocessor in `models/scaler.joblib`.
  - 50 ordered sequences exported to `data/processed/sequences/sequences_k20.csv` and SPMF text files.
- **Next Stage (Member 4):**
  - Use `hallucinated_sequences_spmf.txt` and `truthful_sequences_spmf.txt` to run **PrefixSpan Sequential Pattern Mining** (`05_sequence_mining.ipynb`), calculating support, confidence, and contrast ratios to isolate hallucination-predictive patterns.
